# 🔬 Notebook 5d: Ablation 4 — Full-Run Segmentation Head Freezing
This notebook runs **Ablation 4**: Knowledge Distillation on DeepCrack with **Segmentation Head Frozen throughout the ENTIRE training run** (not just Stage 1).
* **Goal**: Compare full-run head freezing against 2-stage progressive unfreezing.
* **Input Dataset**: DeepCrack + pre-computed SAM 2 teacher logits.


In [ ]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box runs


In [ ]:
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas


In [ ]:
import os, shutil
from pathlib import Path
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists(): input_dir = Path("/kaggle/input")
datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)

for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "train_img" in dirs:
        dest = datasets_dir / "deepcrack"
        if os.path.lexists(dest): os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
        os.symlink(root_path, dest)
        print(f"Linked DeepCrack: {root_path} -> {dest}")
        break


In [ ]:
# Run Ablation 4 (Full-Run SegHead Frozen)
import sys
sys.path.insert(0, ".")
from distillation.kd_trainer import KDSegmentationTrainer
from utils.config_loader import load_config, override_config

cfg = load_config("configs/config.yaml")
cfg = override_config(cfg, {
    "project.name": "crack_distill",
    "project.experiment": "ablation_seghead_frozen",
    "distillation.enabled": True,
    "distillation.progressive.enabled": True,
    "distillation.progressive.freeze_head": True,
    "distillation.progressive.unfreeze_epoch_ratio": 1.0,  # Never unfreeze during run
    "distillation.losses.mask_kd.enabled": True,
    "distillation.losses.feature.enabled": True,
    "distillation.losses.boundary.enabled": True,
    "teacher.logits_dir": "data/teacher_logits_box/"
})

trainer = KDSegmentationTrainer(cfg)
trainer.train()
print("✓ Ablation 4 (Full-Run SegHead Frozen) completed!")
